#Transfer Learning

In [1]:
import zipfile
zip_path = '/content/animal_data.zip'

with zipfile.ZipFile(zip_path,'r') as zip_ref:
   zip_ref.extractall('/content')

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [3]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_data = datagen.flow_from_directory("/content/animal_data", target_size=img_size, batch_size=batch_size, class_mode="categorical", subset="training")
val_data = datagen.flow_from_directory( "/content/animal_data", target_size=img_size, batch_size=batch_size, class_mode="categorical", subset="validation")

Found 1574 images belonging to 16 classes.
Found 386 images belonging to 16 classes.


In [4]:
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
output = Dense(16, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [5]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [6]:
history = model.fit(train_data, validation_data=val_data, epochs=5)

Epoch 1/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 52s 667ms/step - accuracy: 0.6836 - loss: 1.1778 - val_accuracy: 0.8316 - val_loss: 0.5341
Epoch 2/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 67ms/step - accuracy: 0.9091 - loss: 0.3106 - val_accuracy: 0.8756 - val_loss: 0.4201
Epoch 3/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.9644 - loss: 0.1585 - val_accuracy: 0.8860 - val_loss: 0.4148
Epoch 4/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 70ms/step - accuracy: 0.9803 - loss: 0.0938 - val_accuracy: 0.8834 - val_loss: 0.3741
Epoch 5/5
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 88ms/step - accuracy: 0.9924 - loss: 0.0566 - val_accuracy: 0.9016 - val_loss: 0.3566


# Transfer earlystop, Dataaugmentation

In [7]:
import zipfile
zip_path = '/content/animal_data.zip'

with zipfile.ZipFile(zip_path,'r') as zip_ref:
   zip_ref.extractall('/content')

In [8]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [9]:
img_size = (224,224)
batch_size = 32

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2, rotation_range=20, zoom_range=0.2, width_shift_range=0.2, height_shift_range=0.2, horizontal_flip=True, fill_mode="nearest")
train_data = datagen.flow_from_directory("/content/animal_data", target_size=img_size, batch_size=batch_size, class_mode="categorical", subset="training")
val_data = datagen.flow_from_directory("/content/animal_data", target_size=img_size, batch_size=batch_size, class_mode="categorical", subset="validation")

Found 1574 images belonging to 16 classes.
Found 386 images belonging to 16 classes.


In [10]:
base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=(224,224,3))

In [11]:
base_model.trainable = False

In [12]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(16, activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=output)

In [13]:
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [14]:
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
checkpoint = ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True)

In [15]:
history = model.fit(train_data, validation_data=val_data, epochs=20, callbacks=[early_stop, checkpoint])

Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 51s 825ms/step - accuracy: 0.5769 - loss: 1.4334 - val_accuracy: 0.7642 - val_loss: 0.7511
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 524ms/step - accuracy: 0.8024 - loss: 0.6221 - val_accuracy: 0.8187 - val_loss: 0.5764
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 525ms/step - accuracy: 0.8488 - loss: 0.4848 - val_accuracy: 0.7798 - val_loss: 0.6356
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 531ms/step - accuracy: 0.8761 - loss: 0.3930 - val_accuracy: 0.8420 - val_loss: 0.5596
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 27s 541ms/step - accuracy: 0.8964 - loss: 0.3126 - val_accuracy: 0.8679 - val_loss: 0.4779
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 524ms/step - accuracy: 0.9066 - loss: 0.3164 - val_accuracy: 0.8316 - val_loss: 0.5206
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 527ms/step - accuracy: 0.8977 - loss: 0.3147 - val_accuracy: 0.8394 - val_loss: 0.5148
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 522ms/step - accuracy: 0.9206 - loss: 0.2657 - val_accu

In [16]:
base_model.trainable = True

In [17]:
for layer in base_model.layers[:-30]:
    layer.trainable = False

In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])

In [19]:
history_fine = model.fit(train_data, validation_data=val_data, epochs=10, callbacks=[early_stop, checkpoint])

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 61s 858ms/step - accuracy: 0.8361 - loss: 0.5546 - val_accuracy: 0.8627 - val_loss: 0.4239
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 524ms/step - accuracy: 0.8494 - loss: 0.4688 - val_accuracy: 0.8342 - val_loss: 0.4895
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 527ms/step - accuracy: 0.8844 - loss: 0.3792 - val_accuracy: 0.8394 - val_loss: 0.4979
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 26s 514ms/step - accuracy: 0.8996 - loss: 0.3447 - val_accuracy: 0.8135 - val_loss: 0.5694
